# Code Mode Agents - Data Analysis

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/code-mode-analysis/code-mode-analysis-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Most tool using agents call one tool at a time. The model asks for a tool, reads the result, reasons, asks for the next one. Every intermediate result travels back through the context window.

A **code mode** agent writes one program instead. It calls the tools, loops, compares, and aggregates, all in a single block of Python. That program runs in [Monty](https://github.com/pydantic/monty), a sandbox with no imports, no filesystem, and no network, where the only things it can touch are the tools you registered. Only the answer comes back to the model.

The part you cannot get from a generic code interpreter: **those tools can be Flyte tasks**. So a loop in the model's generated code becomes a fan-out of durable, retryable, parallel containers.

We build that up in five steps, against real data: the NYC Yellow Taxi trip records, about 3 million trips a month, queried with DuckDB.

| Step | What it shows |
|---|---|
| 0 | Download the dataset. No API key needed. |
| 1 | The sandbox, with no LLM anywhere. No API key needed. |
| 2 | The model writes the program. The whole loop, hand rolled. |
| 3 | The real agent. Generated loops become parallel durable tasks. |
| 4 | Sequential tool calling vs code mode, measured. |
| 5 | Serve it as a chat app. |

Each step runs the workflow first, renders its report, then shows the code that produced it.

---

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/code-mode-analysis
    !uv pip install -r requirements.txt
    !uv pip install keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file

### Set your Anthropic key

Steps 2 through 5 call a model. Steps 0 and 1 do not, so you can run those with no key at all.

Locally you can also put `ANTHROPIC_API_KEY=sk-ant-...` in a `.env` file: `config.py` loads it for you.

In [ ]:
# Set the Anthropic key. Skip this if it's already in a .env or your environment , 
# config.py calls load_dotenv() for you.
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

In [ ]:
# Optional: connect to a Flyte or Union cluster.
# Skip this to run everything locally. Steps still work, you just do not get
# containers or a real fan-out.
#
# Don't have a cluster? Request demo access at https://union.ai/
#
# !flyte create config \
#     --endpoint <your-endpoint> \
#     --project flytesnacks \
#     --domain development \
#     --builder remote
#
# Then create the secret the tasks read, in the SAME project and domain:
# !flyte create secret ANTHROPIC_API_KEY -p flytesnacks -d development

### Running the workflows

Every step is a plain `flyte run`, so the usual flags work:

| | |
|---|---|
| `flyte run --local ...` | Runs on this machine. No cluster, no image build. Still gets caching and retries. |
| `flyte run --local --tui ...` | Same, with the live task UI. |
| `flyte run ...` | Runs on the cluster: containers, fan-out, retries, and the run graph in the UI. |

The cells below use `--local` so the notebook works with no cluster at all.

One thing to keep in mind: `flyte_map` runs **sequentially** under `--local`, and Flyte prints a warning saying so. The generated code and the answer are identical either way, but the parallel containers in step 3 only happen on a cluster.

### 0. Download the dataset

Pulls the NYC taxi months into blob storage. Optional, since months are fetched on demand anyway, but running it once up front means nobody's first question waits on a download.

`fetch_trips` is cached on the month, not on the SQL, so each month is fetched from the TLC exactly once and served from storage to every query after that.

In [ ]:
!flyte run --local step0_download_data.py download

In [ ]:
view_file("step0_download_data.py", title="the download step", show_path=True)
view_file("dataset.py", title="the dataset: URLs, schema, and the description the model reads", show_path=True)

### 1. The sandbox, with no LLM anywhere

Before letting a model write code, look at what runs it.

`@env.sandbox.orchestrator` marks a task whose body executes inside Monty, a Python interpreter with no imports, no filesystem, and no network. It loops over months and calls `load_month`. Each time it does, Monty pauses, Flyte runs that task in a real container (which does have network access), and Monty resumes with the result.

That pause, dispatch, resume is the whole mechanism. Everything else in this tutorial follows from it.

In [ ]:

!flyte run --local step1_sandbox.py monthly_tip_trend

In [ ]:
# The orchestrator body is pure control flow. The only way it reaches the outside
# world is by calling load_month, which Flyte dispatches for it.
view_file("step1_sandbox.py", title="the sandboxed orchestrator", show_path=True)

### 2. The model writes the program

Same sandbox, except the program arrives as a string. That means it does not have to exist until runtime, and does not have to be written by a human.

This step hand rolls the whole loop in about forty lines: ask the model for a program, run it in Monty with the tools bound, and if it raises, hand the error back and let the model fix its own code. Step 3 hides all of this behind a class, so it is worth seeing the moving parts once.

The task is decorated `@env.task(report=True)`, so it writes an HTML report. We render it below.

In [ ]:
!flyte run --local step2_generated_code.py analyze \
    --question "Did tipping change between January and December 2024?"

In [ ]:
# The report: charts, metrics, and the program the model actually wrote.
from report import show_latest

show_latest()

In [ ]:
# The generate, execute, retry loop. Note the system prompt: the tool docs are
# generated from the signatures and docstrings in tools.py, so adding a tool needs
# no other change.
view_file("step2_generated_code.py", title="the hand rolled loop", show_path=True)

# The tools the model may call. `query` runs guarded, read only SQL. The render
# tools build the report. These docstrings ARE the prompt.
view_file("tools.py", title="the tools", show_path=True)

### 3. The agent, and the fan-out

`Agent(code_mode=True)` runs that loop for you. One thing changes, and it is the important one: `query` is now an `@env.task`. Every query the model writes dispatches through the Flyte controller as a durable child task, retried, cached, and visible in the UI.

So when the model answers a twelve month question by writing `flyte_map("query", sqls, months, concurrency=4)`, that is not a for loop. It is twelve containers, each pulling its own month, each retried independently. The model wrote a fan-out without knowing it wrote a fan-out.

Meanwhile the cheap tools (`create_chart`, `create_metric`) stay in process, where a round trip would only add latency. The model calls them all the same way. Where each one runs is your decision, not its.

> Running `--local`, `flyte_map` executes sequentially, and Flyte says so in a warning. The generated code and the answer are identical, but the twelve parallel containers only happen on a cluster. Drop `--local` and open the run in the UI to see the child tasks.

In [ ]:
!flyte run --local step3_agent_report.py analyze \
    --question "Rank the boroughs by tip rate and show how it moved through 2024"

In [ ]:
# Look for flyte_map in "The program the model wrote" at the bottom of the report.
from report import show_latest

show_latest()

In [ ]:
# The task that renders the report.
view_file("step3_agent_report.py", title="the agent task", show_path=True)

# The durable half. `query` is an @env.task, so the model's queries become child
# tasks; the render helpers are plain functions and run in process. Same tools list,
# two very different execution paths. The instructions also live here.
view_file("analysis.py", title="the durable query task, the instructions, the agent", show_path=True)

### 4. Why bother? Measure it.

The same question, the same tools, the same model, run twice. The only difference is `code_mode=False` versus `code_mode=True`.

Sequential tool calling emits one JSON tool call, reads the result, thinks, emits the next one. Every intermediate result, every row of every query, travels back through the context window, and every step costs a round trip. Code mode writes one program: the loop runs in the sandbox, and only the answer comes back.

Watch the turn count. Sequential grows with the work; code mode does not. The report below charts turns and tokens side by side, and prints both answers so you can judge them yourself.

This one takes a few minutes, since it runs both agents.

In [ ]:
!flyte run --local step4_compare_modes.py compare \
    --question "Which borough tipped best in each quarter of 2024?"

In [ ]:
# Turns and tokens, side by side: sequential tool calling vs code mode.
from report import show_latest

show_latest()

In [ ]:
# One flag apart. Both agents get the same tools and the same instructions.
view_file("step4_compare_modes.py", title="sequential vs code mode", show_path=True)

# Where the turns and tokens get counted: the LLM callback. This is also the only
# provider specific code in the tutorial.
view_file("llm.py", title="the LLM callback and the usage tally", show_path=True)

### 5. Serve it as a chat app (stretch)

Two front ends, and the difference is the point of step 3.

**Here in the notebook** ,  a small Gradio UI over the same agent. Gradio renders inline in Colab, which is why we use it for the notebook. The agent runs in this process, so there are **no durable child tasks and no real fan-out**. It's the right way to poke at the prompt.

**Deployed** (last cell) ,  the native `AgentChatAppEnvironment`: the chat UI, streaming, and the endpoint in one declaration, and every message becomes a **durable Flyte run** whose `query` calls fan out as child tasks you can click into.

Try:
- *"Do riders in Brooklyn and the Bronx really tip less than Manhattan, or is something else going on?"*
- *"How did the tip rate move month by month through 2024? Chart it."*

In [ ]:
!pip install -q gradio

In [ ]:
# A Gradio front end over the same agent ,  Gradio renders inline in Colab, which is
# why we use it here rather than the native chat app (that one is what gets deployed).
#
# The agent runs in this process, so there are no durable child tasks and no real
# fan-out. Deploy it (next cell) for that.
import asyncio

import gradio as gr

import flyte
import tools
from analysis import build_agent

flyte.init()  # local: tasks run in-process


def ask(question):
    tools.new_report()
    agent, usage = build_agent(code_mode=True)
    result = asyncio.run(agent.run.aio(question))

    program = usage.programs[0] if usage.programs else (result.code or "")
    charts = "".join(tools.collect_report())
    return result.summary or result.error, charts, program


with gr.Blocks(title="NYC Taxi analyst") as demo:
    gr.Markdown(
        "## NYC Taxi analyst\n"
        "Ask a question. Claude writes one Python program, the Monty sandbox runs it, "
        "and the only things it can touch are the tools we registered."
    )
    question = gr.Textbox(
        label="Question",
        placeholder="How did the tip rate move month by month through 2024?",
    )
    ask_btn = gr.Button("Ask", variant="primary")

    answer = gr.Markdown(label="Answer")
    report = gr.HTML(label="Report")
    program = gr.Code(label="The program the model wrote", language="python")

    ask_btn.click(ask, inputs=question, outputs=[answer, report, program])
    question.submit(ask, inputs=question, outputs=[answer, report, program])

    gr.Examples(
        examples=[
            "Do riders in Brooklyn and the Bronx really tip less than Manhattan, "
            "or is something else going on?",
            "How did the tip rate move month by month through 2024? Chart it.",
            "How do JFK and LaGuardia airport pickups compare on trip distance, fare, "
            "and tip rate across 2024?",
        ],
        inputs=question,
    )

demo.launch(share=True)  # Colab renders this inline; share=True also gives a public link

In [ ]:
# Deploy it to the cluster instead ,  needs a Flyte/Union connection.
!python step5_chat_app.py deploy

In [ ]:
# The deployed app. `task_entrypoint=answer` is what makes each message a durable
# run: an app's request handler has no task context, so calling the agent straight
# from it would run the sandboxed queries inside the app pod.
view_file("step5_chat_app.py", title="the chat app", show_path=True)